In [0]:
# Load bronze tables
df_financials = spark.table("bronze_db.bronze_financials")
df_failed = spark.table("bronze_db.bronze_failed_banks")
df_institutions = spark.table("bronze_db.bronze_institutions")
df_summary = spark.table("bronze_db.bronze_summary")

print("bronze_financials  :", df_financials.count(), "rows")
print("bronze_failed_banks:", df_failed.count(), "rows")
print("bronze_institutions:", df_institutions.count(), "rows")
print("bronze_summary     :", df_summary.count(), "rows")

bronze_financials  : 10000 rows
bronze_failed_banks: 4114 rows
bronze_institutions: 10000 rows
bronze_summary     : 7989 rows


In [0]:
from pyspark.sql.functions import col, when, to_date, year, month
from pyspark.sql.types import DoubleType, IntegerType

# Clean financials
df_fin_clean = df_financials \
    .dropDuplicates() \
    .dropna(subset=["CERT", "ASSET", "DEP"]) \
    .withColumn("ASSET", col("ASSET").cast(DoubleType())) \
    .withColumn("DEP", col("DEP").cast(DoubleType())) \
    .withColumn("EQTOT", col("EQTOT").cast(DoubleType())) \
    .withColumn("NETINC", col("NETINC").cast(DoubleType())) \
    .withColumn("LNLSNET", col("LNLSNET").cast(DoubleType())) \
    .withColumn("CERT", col("CERT").cast(IntegerType()))

# Clean institutions
df_inst_clean = df_institutions \
    .dropDuplicates() \
    .dropna(subset=["CERT", "NAME"]) \
    .withColumn("CERT", col("CERT").cast(IntegerType())) \
    .withColumn("ASSET", col("ASSET").cast(DoubleType()))

# Clean failed banks — add ML label column
df_failed_clean = df_failed \
    .dropDuplicates() \
    .dropna(subset=["CERT", "FAILDATE"]) \
    .withColumn("CERT", col("CERT").cast(IntegerType())) \
    .withColumn("failed", when(col("FAILDATE").isNotNull(), 1).otherwise(0))

print("Financials cleaned :", df_fin_clean.count(), "rows")
print("Institutions cleaned:", df_inst_clean.count(), "rows")
print("Failed banks cleaned:", df_failed_clean.count(), "rows")

Financials cleaned : 10000 rows
Institutions cleaned: 10000 rows
Failed banks cleaned: 3626 rows


In [0]:
from pyspark.sql.functions import round

# Create financial ratio features for each bank
df_features = df_fin_clean.withColumn(
    "capital_ratio", round(col("EQTOT") / col("ASSET"), 4)
).withColumn(
    "loan_to_deposit_ratio", round(col("LNLSNET") / col("DEP"), 4)
).withColumn(
    "profit_margin", round(col("NETINC") / col("ASSET"), 4)
).withColumn(
    "deposit_to_asset_ratio", round(col("DEP") / col("ASSET"), 4)
).withColumn(
    "leverage_ratio", round(col("ASSET") / col("EQTOT"), 4)
).na.fill(0)

print("Feature columns added:")
print(df_features.columns)
display(df_features.select(
    "CERT", "ASSET", "capital_ratio", 
    "loan_to_deposit_ratio", "profit_margin", 
    "leverage_ratio"
).limit(5))

Feature columns added:
['REPDTE', 'ASSET', 'CERT', 'NETINC', 'LNLSNET', 'DEP', 'EQTOT', 'ID', 'ingestion_timestamp', 'source', 'layer', 'capital_ratio', 'loan_to_deposit_ratio', 'profit_margin', 'deposit_to_asset_ratio', 'leverage_ratio']


CERT,ASSET,capital_ratio,loan_to_deposit_ratio,profit_margin,leverage_ratio
10016,14130.0,0.1149,0.4079,0.0054,8.7007
10016,14902.0,0.1182,0.4377,0.0048,8.4574
10018,81204.0,0.1381,0.3993,0.019,7.242
10018,82610.0,0.1422,0.4002,0.0078,7.0324
10018,87518.0,0.1536,0.4081,0.0129,6.5093


In [0]:
%pip install networkx

In [0]:

import pandas as pd
import networkx as nx

from pyspark.sql.functions import avg, sum as spark_sum

# Aggregate financial data per bank
df_bank_agg = df_features.groupBy("CERT").agg(
    avg("ASSET").alias("avg_asset"),
    avg("DEP").alias("avg_dep"),
    avg("EQTOT").alias("avg_equity"),
    avg("capital_ratio").alias("avg_capital_ratio"),
    avg("leverage_ratio").alias("avg_leverage_ratio"),
    avg("profit_margin").alias("avg_profit_margin"),
    avg("loan_to_deposit_ratio").alias("avg_loan_to_deposit")
)

# Convert to pandas for network building
pdf_banks = df_bank_agg.toPandas()

# Build network graph
G = nx.Graph()

# Add nodes — each bank is a node
for _, row in pdf_banks.iterrows():
    G.add_node(int(row["CERT"]),
               avg_asset=row["avg_asset"],
               avg_equity=row["avg_equity"],
               capital_ratio=row["avg_capital_ratio"])

# Add edges — connect banks with similar deposit levels
# (proxy for inter-bank relationships)
pdf_sorted = pdf_banks.sort_values("avg_dep", ascending=False).head(500)
for i in range(len(pdf_sorted)):
    for j in range(i+1, min(i+5, len(pdf_sorted))):
        bank_i = int(pdf_sorted.iloc[i]["CERT"])
        bank_j = int(pdf_sorted.iloc[j]["CERT"])
        weight = abs(pdf_sorted.iloc[i]["avg_dep"] - pdf_sorted.iloc[j]["avg_dep"])
        G.add_edge(bank_i, bank_j, weight=weight)

print("Network built!")
print("Total banks (nodes):", G.number_of_nodes())
print("Total connections (edges):", G.number_of_edges())

Network built!
Total banks (nodes): 108
Total connections (edges): 422


In [0]:
# Calculate graph features for each bank node
pagerank = nx.pagerank(G)
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G)
clustering_coef = nx.clustering(G)

# Combine all graph features into a dataframe
graph_features = []
for node in G.nodes():
    graph_features.append({
        "CERT": node,
        "pagerank": pagerank.get(node, 0),
        "degree_centrality": degree_centrality.get(node, 0),
        "betweenness_centrality": betweenness_centrality.get(node, 0),
        "clustering_coefficient": clustering_coef.get(node, 0)
    })

pdf_graph = pd.DataFrame(graph_features)
df_graph_features = spark.createDataFrame(pdf_graph)

print("Graph features created for", df_graph_features.count(), "banks")
display(df_graph_features.limit(5))

Graph features created for 108 banks


CERT,pagerank,degree_centrality,betweenness_centrality,clustering_coefficient
10016,0.007655095118760127,0.07476635514018691,0.018155535846875334,0.6428571428571429
10018,0.006493343990497552,0.07476635514018691,0.12291817413458675,0.6428571428571429
1001,0.007929263874124423,0.07476635514018691,0.12394002775821834,0.6428571428571429
10025,0.012038401616218346,0.07476635514018691,0.049977719153623194,0.6428571428571429
10026,0.0100525304464282,0.07476635514018691,0.12212270882972999,0.6428571428571429


In [0]:
# Join financial features with graph features
df_silver = df_features.join(df_graph_features, on="CERT", how="left")

# Add failed label (1 = failed, 0 = active)
failed_certs = df_failed_clean.select("CERT").distinct()
df_silver = df_silver.join(failed_certs, on="CERT", how="left") \
    .withColumn("failed", when(col("CERT").isin(
        [row.CERT for row in failed_certs.collect()]
    ), 1).otherwise(0))

print("Silver dataset shape:", df_silver.count(), "rows,", len(df_silver.columns), "columns")
display(df_silver.limit(5))

Silver dataset shape: 10000 rows, 21 columns


CERT,REPDTE,ASSET,NETINC,LNLSNET,DEP,EQTOT,ID,ingestion_timestamp,source,layer,capital_ratio,loan_to_deposit_ratio,profit_margin,deposit_to_asset_ratio,leverage_ratio,pagerank,degree_centrality,betweenness_centrality,clustering_coefficient,failed
10016,19900630,14130.0,76.0,5066.0,12419.0,1624.0,10016_19900630,2026-03-12T18:22:25.561Z,FDIC_API,bronze,0.1149,0.4079,0.0054,0.8789,8.7007,0.007655095118760127,0.07476635514018691,0.018155535846875334,0.6428571428571429,0
10016,19910630,14902.0,72.0,5698.0,13019.0,1762.0,10016_19910630,2026-03-12T18:22:25.561Z,FDIC_API,bronze,0.1182,0.4377,0.0048,0.8736,8.4574,0.007655095118760127,0.07476635514018691,0.018155535846875334,0.6428571428571429,0
10018,19851231,81204.0,1544.0,26887.0,67332.0,11213.0,10018_19851231,2026-03-12T18:22:25.561Z,FDIC_API,bronze,0.1381,0.3993,0.019,0.8292,7.242,0.006493343990497552,0.07476635514018691,0.12291817413458675,0.6428571428571429,0
10018,19860630,82610.0,647.0,27522.0,68764.0,11747.0,10018_19860630,2026-03-12T18:22:25.561Z,FDIC_API,bronze,0.1422,0.4002,0.0078,0.8324,7.0324,0.006493343990497552,0.07476635514018691,0.12291817413458675,0.6428571428571429,0
10018,19870930,87518.0,1129.0,29420.0,72097.0,13445.0,10018_19870930,2026-03-12T18:22:25.561Z,FDIC_API,bronze,0.1536,0.4081,0.0129,0.8238,6.5093,0.006493343990497552,0.07476635514018691,0.12291817413458675,0.6428571428571429,0


In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_silver.withColumn("ingestion_timestamp", current_timestamp()) \
         .withColumn("layer", lit("silver")) \
         .write.format("delta") \
         .mode("overwrite") \
         .option("mergeSchema", "true") \
         .saveAsTable("silver_db.silver_bank_features")

print("silver_bank_features saved:", spark.table("silver_db.silver_bank_features").count(), "rows")

silver_bank_features saved: 10000 rows
